# Encapsulation Mechanics and Aggregation Architecture

**Session:** 2026-11-03 · OOP Practice Lab  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

In object-oriented software engineering, two foundational architectural concepts govern how objects interact:
1. **Data Encapsulation:** Unifying data (attributes) and behavior (methods) while restricting direct external access to internal state through access modifiers, compiler name mangling, and property descriptors.
2. **Object Aggregation:** Establishing a structural relationship where one object references another independent object (a *has-a* association) without taking ownership of its lifecycle.

In this notebook, we implement and verify:
- Class definition, state initialization, and instance isolation.
- Public, protected, and private access modifiers.
- Python compiler name mangling (`_ClassName__attribute`).
- Modern property descriptors (`@property`, `@setter`).
- **Exercise 1:** The `Manufacturer` and `Device` aggregation model, verifying shared lifecycles and loose coupling against tight composition.


## 1. Class and Object Fundamentals

A class serves as an abstract blueprint, whereas an object is a stateful instance allocated in memory. Every instance maintains its own data members while sharing method implementations.


In [ ]:
class HardwareComponent:
    """Demonstrates fundamental class blueprint and instance state isolation."""

    def __init__(self, serial_number: str, clock_speed_ghz: float) -> None:
        self.serial_number: str = serial_number
        self.clock_speed_ghz: float = float(clock_speed_ghz)
        self.power_state: bool = False

    def power_on(self) -> None:
        """Mutate the internal power state."""
        self.power_state = True

    def power_off(self) -> None:
        """Power off the hardware component."""
        self.power_state = False

    def describe(self) -> str:
        """Return a descriptive summary string of component state."""
        status = "ON" if self.power_state else "OFF"
        return f"Component [SN: {self.serial_number}, Clock: {self.clock_speed_ghz:.2f} GHz, Status: {status}]"


# Verification of instance state isolation
c1 = HardwareComponent("HW-101", 3.2)
c2 = HardwareComponent("HW-102", 4.0)

c1.power_on()
assert c1.power_state is True
assert c2.power_state is False  # c2 state remains completely isolated
print("HardwareComponent isolation verified successfully.")

## 2. Access Modifiers and Name Mangling

Python provides three conventional access tiers:
- **Public (`self.x`):** Freely accessible and modifiable by any external caller.
- **Protected (`self._x`):** Signals an internal implementation detail meant for the class and its derived subclasses.
- **Private (`self.__x`):** Triggers compiler name mangling to `_ClassName__x` to prevent accidental overriding and signal strict private encapsulation.


In [ ]:
class SecureAccount:
    """Demonstrates public, protected, and private access tiers with name mangling."""

    def __init__(self, account_id: str, secret_key: str, initial_credit: float) -> None:
        self.account_id: str = account_id  # Public
        self._secret_key: str = secret_key  # Protected
        self.__balance: float = float(initial_credit)  # Private (mangled)

    def get_balance(self) -> float:
        """Controlled public accessor for private balance."""
        return self.__balance

    def adjust_balance(self, delta: float) -> None:
        """Mutator enforcing business invariants."""
        if self.__balance + delta < 0.0:
            raise ValueError("Insufficient credit balance.")
        self.__balance += delta


acc = SecureAccount("ACC-9001", "sk_test_987", 500.0)
assert acc.account_id == "ACC-9001"
assert acc._secret_key == "sk_test_987"
assert acc.get_balance() == 500.0

# Verify compiler name mangling in __dict__
assert "_SecureAccount__balance" in acc.__dict__
assert acc._SecureAccount__balance == 500.0
print("Name mangling verified:", [k for k in acc.__dict__ if "__" in k])

## 3. Property Descriptors for Defensive Encapsulation

The `@property` decorator allows attributes to be read and validated using standard dot syntax without exposing raw internal variables.


In [ ]:
class VoltageRegulator:
    """Demonstrates defensive bounds checking with Python property descriptors."""

    def __init__(self, target_voltage: float) -> None:
        self._target_voltage: float = 0.0
        self.target_voltage = target_voltage  # Triggers property setter validation

    @property
    def target_voltage(self) -> float:
        """Return the regulated target voltage in Volts."""
        return self._target_voltage

    @target_voltage.setter
    def target_voltage(self, value: float) -> None:
        """Enforce operating range between 0.8V and 5.0V."""
        if not (0.8 <= value <= 5.0):
            raise ValueError(
                f"Target voltage {value}V out of safe operational envelope [0.8V, 5.0V]."
            )
        self._target_voltage = float(value)


reg = VoltageRegulator(1.2)
assert reg.target_voltage == 1.2
reg.target_voltage = 3.3
assert reg.target_voltage == 3.3

try:
    reg.target_voltage = 12.0
    raise AssertionError("Failed to catch voltage violation.")
except ValueError as e:
    print("Caught expected safety violation:", e)

## 4. Exercise 1: Aggregation Architecture (`Manufacturer` & `Device`)

In Exercise 1, we model the relationship between an industrial `Manufacturer` and a commercial `Device`.
- Under **Aggregation**, a `Device` holds an external reference to a `Manufacturer`. Multiple devices can share the same manufacturer instance without lifecycle coupling.
- Under **Composition**, a `Device` would instantiate and own its internal manufacturer instance, tying their lifecycles together.


In [ ]:
class Manufacturer:
    """Represents an industrial hardware manufacturer entity."""

    def __init__(self, identity: int, location: str) -> None:
        self.__identity: int = int(identity)
        self.__location: str = str(location)

    def get_identity(self) -> int:
        """Return the manufacturer unique identity code."""
        return self.__identity

    def get_location(self) -> str:
        """Return the geographical facility location."""
        return self.__location

    def describe(self) -> str:
        """Output manufacturer information summary."""
        msg = f"Manufacturer ID: {self.__identity}, Location: {self.__location}"
        print(msg)
        return msg


class Device:
    """Represents a computing hardware device aggregating a Manufacturer reference."""

    def __init__(self, name: str, price: float, manufacturer: Manufacturer) -> None:
        self.__name: str = str(name)
        self.__price: float = float(price)
        if not isinstance(manufacturer, Manufacturer):
            raise TypeError(f"Expected Manufacturer instance, got {type(manufacturer).__name__}")
        self.__manufacturer: Manufacturer = manufacturer

    def get_name(self) -> str:
        """Return the device commercial name."""
        return self.__name

    def get_price(self) -> float:
        """Return the unit price."""
        return self.__price

    def get_manufacturer(self) -> Manufacturer:
        """Return the aggregated manufacturer instance."""
        return self.__manufacturer

    def describe(self) -> str:
        """Print and return the device description, delegating to the aggregated manufacturer."""
        mfg_desc = self.__manufacturer.describe()
        device_msg = f"Device Name: {self.__name}, Price: ${self.__price:.2f} | {mfg_desc}"
        print(device_msg)
        return device_msg


# Test aggregation: Single manufacturer instance shared across multiple devices
shared_mfg = Manufacturer(identity=101, location="Munich, Germany")
device_a = Device("EdgeNode-X1", 1299.99, shared_mfg)
device_b = Device("SmartGateway-Mini", 199.95, shared_mfg)

# Verify both devices reference the EXACT same instance in memory
assert device_a.get_manufacturer() is device_b.get_manufacturer()
assert device_a.get_manufacturer().get_location() == "Munich, Germany"
print("Aggregation test passed: single manufacturer shared across multiple devices.")

## 5. Ingesting Real Data & Lifecycle Verification

We now load the tabular catalog in `../data/device_catalog.csv` to instantiate a fleet of devices and verify independent object lifecycles.


In [ ]:
import csv
from pathlib import Path

csv_path = Path("../data/device_catalog.csv")
assert csv_path.is_file(), f"Missing dataset: {csv_path}"

# Cache manufacturers to demonstrate aggregation reuse
manufacturers: dict[int, Manufacturer] = {}
devices: list[Device] = []

with csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        mfg_id = int(row["manufacturer_id"])
        country = row["manufacturer_country"]
        if mfg_id not in manufacturers:
            manufacturers[mfg_id] = Manufacturer(identity=mfg_id, location=country)

        device = Device(
            name=row["device_name"],
            price=float(row["price_usd"]),
            manufacturer=manufacturers[mfg_id],
        )
        devices.append(device)

print(
    f"Instantiated {len(devices)} devices aggregating {len(manufacturers)} distinct manufacturers."
)
assert len(devices) == 7
assert len(manufacturers) == 4

# Test lifecycle independence: deleting devices leaves manufacturer catalog intact
first_dev_mfg = devices[0].get_manufacturer()
del devices

assert first_dev_mfg.get_identity() == 101
print("Lifecycle independence verified: deleting device fleet does not destroy manufacturers.")